In [ ]:
#Projeto para analise/monitoramento dos principais KPIs de uma empresa financeira
#Base de dados utilizada: Kaggle Company Financials Dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rich.jupyter import display

In [ ]:
df = pd.read_csv('Financials.csv')

In [ ]:
#Inspecionando a Base de dados
df

In [ ]:
#Principais dados de interesse:

#Gross Sales = receita bruta, nesse caso preço x quantidade vendida
#sales = Receita da venda após aplicação do desconto, nesse caso RECEITA LÍQUIDA
#Profit = Lucro, nesse caso gross sales - COGS
#COGS = Custos da mercadoria vendida, cada tipo de produto tem um custo associado (e não mostrado nessa database). A coluna manufacturing price, não nos diz nada.


In [ ]:
#Tratando a base de dados:

#Removendo espaços em brancos nos nomes das colunas
df.columns = df.columns.str.strip()

#Remover o cifrão '$', espaços e vírgulas, transformando em número
for col in ['Units Sold', 'Sale Price', 'Gross Sales', 'Discounts', 'Sales', 'COGS', 'Profit']:
    # Transforma em texto para garantir, remover o '$' e limpar espaços
    df[col] = df[col].astype(str).str.replace('$', '', regex=False).str.strip()
    # Remover as vírgulas que separam os milhares (ex: 1,618.50 virando 1618.50)
    df[col] = df[col].str.replace(',', '', regex=False)
    # Converter para número decimal
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
df

In [ ]:
#Utilizando a engenharia de atributos:
#Precisamos primeiro listar todos os custos associados, por tipo de produto. Para essa database podemos dividir o COGS pelo Units Sold
df['Cost Manufacturing'] = df['COGS'] / df['Units Sold']
#Isso nos dá o custo real para produzir uma unidade do detmerinado produto, podemos agora descartar o Manufacturing Price de nossos cálculos

In [ ]:
df

In [ ]:
#Podemos agora começar a trabalhar com nossos KPIS:

#KPIS de Receita e Lucro:
#Margem de lucro bruto:
df['Gross Profit Margin'] = df['Profit'] / df['Sales']
df['Gross Profit Margin'] = df['Gross Profit Margin'].map('{:.2%}'.format)

In [ ]:
df

In [ ]:
#KPIS de Receita e Lucro:
#Margem de Contribuição por Produto:
df['Unit Contribution Margin'] = (df['Sale Price'] - df['Cost Manufacturing']) / df['Sale Price']
df['Unit Contribution Margin'] = df['Unit Contribution Margin'].map('{:.2%}'.format)

In [ ]:
df

In [ ]:
#KPIS de Receita e Lucro:
#Lucro Total por mês:
lucro_total_mes = df.groupby(['Month Number', 'Month Name'])['Profit'].sum()


In [ ]:
lucro_total_mes

In [ ]:
#KPIS de Vendas e Performance:
#Ticket Médio por produto
ticket_por_produto = (df.groupby('Product', as_index=True).agg({'Sales': 'sum','Units Sold': 'sum'}))

ticket_por_produto['Average Ticket'] = (ticket_por_produto['Sales'] / ticket_por_produto['Units Sold'])

In [ ]:
ticket_por_produto

In [ ]:
#KPIS de Vendas e Performance:
#Volume de Vendas (TPV) por mês:
tpv_mensal = df.groupby(['Month Number', 'Month Name'])['Units Sold'].sum()

In [ ]:
tpv_mensal

In [ ]:
#KPIS de Vendas e Performance:
#Receita Líquida por mês (Net Sales por mês):
net_sales_mensal = df.groupby(['Month Number', 'Month Name'])['Sales'].sum()

In [ ]:
net_sales_mensal

In [ ]:
#KPIS de Eficiência Comercial:
#Como a coluna possun valors nulos, vamos substituir por zeros (que faz mais sentido nesse estudo). Pois sem desconto significa desconto = 0
df['Discounts'] = df['Discounts'].fillna(0)

#Taxa de Desconto:
df['Discount Impact'] = (df['Discounts'] / df['Gross Sales'])
df['Discount Impact'] = df['Discount Impact'].map('{:.2%}'.format)


In [ ]:
df

In [ ]:
#KPIS de Eficiência Comercial:

#Custo da Mercadoria vendida:
df['Cogs Sales'] = (df['COGS'] / df['Sales'])
df['Cogs Sales'] = df['Cogs Sales'].map('{:.2%}'.format)

In [ ]:
df

In [ ]:
#KPIS de Crescimento e Mercado:

#Crescimento Mensal (MoM):
vendas_mensais = df.groupby(['Month Number', 'Month Name'])['Sales'].sum().reset_index()
vendas_mensais['MoM Growth (%)'] = vendas_mensais['Sales'].pct_change() * 100

In [ ]:
vendas_mensais

In [ ]:
#KPIS de Crescimento e Mercado:

#Receita Total por País (Market Sharre)
receita_por_pais = (df.groupby('Country', as_index=False)['Sales'].sum())

In [ ]:
receita_por_pais

In [ ]:
#KPIS de Crescimento e Mercado:

#Mix de produtos (Qual produto mais rentável)

mix_produtos = df.groupby('Product')['Profit'].sum().reset_index()
lucro_total = mix_produtos['Profit'].sum()
mix_produtos['Mix (%)'] = (mix_produtos['Profit'] / lucro_total) * 100
mix_produtos = mix_produtos.sort_values(by='Mix (%)', ascending=False)

In [ ]:
mix_produtos
